# Lab 2: Uninformed Search — BFS and DFS on a 2D Maze

**Goal:** implement BFS and DFS from scratch, run both on the same maze, and compare path length and maximum frontier size.

`0` = open cell, `1` = wall. Run cells in order.

## 1. The Maze

In [1]:
maze = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 1, 1, 1, 0]
]

start = (0, 0)
goal = (4, 4)

for row in maze:
    print(row)

[0, 0, 0, 0, 0]
[0, 1, 1, 1, 0]
[0, 0, 0, 1, 0]
[0, 1, 0, 0, 0]
[0, 1, 1, 1, 0]


## 2. Movements

In [2]:
directions = [
    (-1, 0),   # up
    (1, 0),    # down
    (0, -1),   # left
    (0, 1)     # right
]

## 3. Checking a Valid Move
A move is valid only if the new cell is inside the maze and not a wall.

In [3]:
def valid_move(maze, row, col):
    rows = len(maze)
    cols = len(maze[0])
    return (
        0 <= row < rows
        and 0 <= col < cols
        and maze[row][col] == 0
    )

# Quick check
print(valid_move(maze, 0, 0))   # True — open cell
print(valid_move(maze, 1, 1))   # False — wall
print(valid_move(maze, -1, 0))  # False — outside the maze

True
False
False


## 4. Breadth-First Search (BFS)
BFS uses a **queue (FIFO)**: `frontier.popleft()` expands the oldest discovered state first, so it explores level by level.

In [4]:
from collections import deque

def bfs(maze, start, goal):
    frontier = deque([(start, [start])])
    visited = {start}
    max_frontier = 1

    while frontier:
        max_frontier = max(max_frontier, len(frontier))

        current, path = frontier.popleft()   # FIFO: oldest first

        if current == goal:
            return path, max_frontier

        row, col = current
        for dr, dc in directions:
            nr, nc = row + dr, col + dc
            neighbor = (nr, nc)

            if valid_move(maze, nr, nc) and neighbor not in visited:
                visited.add(neighbor)
                frontier.append((neighbor, path + [neighbor]))

    return None, max_frontier

In [5]:
bfs_path, bfs_frontier = bfs(maze, start, goal)

print("BFS path:", bfs_path)
print("BFS path length:", len(bfs_path) - 1)
print("BFS maximum frontier:", bfs_frontier)

BFS path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3), (3, 4), (4, 4)]
BFS path length: 8
BFS maximum frontier: 3


`len(path) - 1` is the number of moves, since a path of k states contains k − 1 moves of cost 1 each.

## 5. Depth-First Search (DFS)
DFS uses a **stack (LIFO)**: `frontier.pop()` expands the most recently discovered state first, so it commits to one branch before backtracking.

In [6]:
def dfs(maze, start, goal):
    frontier = [(start, [start])]   # plain list used as a stack
    visited = {start}
    max_frontier = 1

    while frontier:
        max_frontier = max(max_frontier, len(frontier))

        current, path = frontier.pop()   # LIFO: most recent first

        if current == goal:
            return path, max_frontier

        row, col = current
        for dr, dc in directions:
            nr, nc = row + dr, col + dc
            neighbor = (nr, nc)

            if valid_move(maze, nr, nc) and neighbor not in visited:
                visited.add(neighbor)
                frontier.append((neighbor, path + [neighbor]))

    return None, max_frontier

In [7]:
dfs_path, dfs_frontier = dfs(maze, start, goal)

print("DFS path:", dfs_path)
print("DFS path length:", len(dfs_path) - 1)
print("DFS maximum frontier:", dfs_frontier)

DFS path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]
DFS path length: 8
DFS maximum frontier: 3


## 6. Visualizing the Paths
`*` = solution path, `#` = wall, `.` = open cell.

In [8]:
def print_maze(maze, path=None):
    path = set(path or [])
    for r in range(len(maze)):
        row = ""
        for c in range(len(maze[0])):
            if (r, c) in path:
                row += "* "
            elif maze[r][c] == 1:
                row += "# "
            else:
                row += ". "
        print(row)

In [9]:
print("BFS solution:")
print_maze(maze, bfs_path)

print()

print("DFS solution:")
print_maze(maze, dfs_path)

BFS solution:
* . . . . 
* # # # . 
* * * # . 
. # * * * 
. # # # * 

DFS solution:
* * * * * 
. # # # * 
. . . # * 
. # . . * 
. # # # * 


## 7. Compare BFS and DFS

In [10]:
import pandas as pd
comparison = pd.DataFrame({
    "Algorithm": ["BFS", "DFS"],
    "Path Length": [len(bfs_path) - 1, len(dfs_path) - 1],
    "Maximum Frontier": [bfs_frontier, dfs_frontier],
})
comparison

,Algorithm,Path Length,Maximum Frontier
0,BFS,8,3
1,DFS,8,3


**Answers (start (0,0), goal (4,4)):**

1. **Which found the shorter path?** Neither. Both paths are 8 moves, so this is a tie.
2. **Which had the larger maximum frontier?** They are equal (3 each) on this maze.
3. **Why different paths?** The maze has two routes of equal length around the central wall block: down column 0 then across rows 2–3, or along the top row then down column 4. BFS expands in FIFO order and returns the first shortest route it reaches (down column 0). DFS with the order up, down, left, right pushes *right* last, so it pops *right* first and commits to the top row and right column.
4. **Does DFS always find the shortest path?** No. DFS returns the first path it reaches while going deep, not the shortest. In Experiment 2, for (0,0)→(2,2), DFS returns 10 moves while BFS returns 4.
5. **Does BFS always find the shortest path when every move costs 1?** Yes. BFS expands states in order of increasing depth, so the first time it reaches the goal, no path with fewer moves exists.

## 8. Experiment 1 — Change the Neighbor Order
DFS's exact path depends on the order in which neighbors are tried. We re-run DFS (and BFS, for comparison) with the order right, down, left, up.

In [11]:
directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]   # right, down, left, up

dfs_path2, dfs_frontier2 = dfs(maze, start, goal)
bfs_path2, bfs_frontier2 = bfs(maze, start, goal)

print("DFS path (new order):", dfs_path2)
print("DFS path length:", len(dfs_path2) - 1, "| max frontier:", dfs_frontier2)
print("BFS path (new order):", bfs_path2)
print("BFS path length:", len(bfs_path2) - 1, "| max frontier:", bfs_frontier2)
print()
print("DFS (new order):"); print_maze(maze, dfs_path2)
print()
print("BFS (new order):"); print_maze(maze, bfs_path2)

# Reset to the original order before Experiment 2
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right

DFS path (new order): [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3), (3, 4), (4, 4)]
DFS path length: 8 | max frontier: 3
BFS path (new order): [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]
BFS path length: 8 | max frontier: 3

DFS (new order):
* . . . . 
* # # # . 
* * * # . 
. # * * * 
. # # # * 

BFS (new order):
* * * * * 
. # # # * 
. . . # * 
. # . . * 
. # # # * 


**Answer:** Yes, the DFS path changed. With the new order it takes the route down column 0 (the route BFS took before), while the length stayed 8. BFS's path *length* never depends on the order, because it is always the minimum. Its *route* can still change when several shortest routes tie: with the new order BFS returns the top-row route. DFS's length and route both depend on the order, because it commits to whichever branch it tries first.

## 9. Experiment 2 — Change `start` and `goal`
Four different (start, goal) pairs on the same maze. Every cell chosen is open (`0`).

In [12]:
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right

pairs = [((0, 0), (2, 2)), ((3, 2), (0, 0)), ((4, 0), (0, 4)), ((0, 0), (0, 2))]
rows = []
for s, g in pairs:
    b_path, b_f = bfs(maze, s, g)
    d_path, d_f = dfs(maze, s, g)
    rows.append({
        "Start": s, "Goal": g,
        "BFS Length": len(b_path) - 1, "BFS Max Frontier": b_f,
        "DFS Length": len(d_path) - 1, "DFS Max Frontier": d_f,
        "Same path?": b_path == d_path,
    })
pd.DataFrame(rows)

,Start,Goal,BFS Length,BFS Max Frontier,DFS Length,DFS Max Frontier,Same path?
0,"(0, 0)","(2, 2)",4,3,10,3,False
1,"(3, 2)","(0, 0)",5,4,9,3,False
2,"(4, 0)","(0, 4)",8,3,10,3,False
3,"(0, 0)","(0, 2)",2,3,2,2,True


In [13]:
# Visual check of the pair (0,0) -> (2,2)
b_path, _ = bfs(maze, (0, 0), (2, 2))
d_path, _ = dfs(maze, (0, 0), (2, 2))
print("BFS:"); print_maze(maze, b_path); print()
print("DFS:"); print_maze(maze, d_path)

BFS:
* . . . . 
* # # # . 
* * * # . 
. # . . . 
. # # # . 

DFS:
* * * * * 
. # # # * 
. . * # * 
. # * * * 
. # # # . 


In [14]:
# Q9: make the goal unreachable by sealing (4,4) behind walls at (3,4) and (2,4)
sealed = [row[:] for row in maze]
sealed[3][4] = 1
sealed[2][4] = 1
print("BFS on sealed maze:", bfs(sealed, (0, 0), (4, 4)))
print("DFS on sealed maze:", dfs(sealed, (0, 0), (4, 4)))

BFS on sealed maze: (None, 3)
DFS on sealed maze: (None, 2)


**Answers:**

6. **Does the path length change on re-run with the same start/goal?** No, for either. Both algorithms are deterministic here (fixed neighbor order, no randomness), so repeated runs return identical paths.
7. **Effect of distance on maximum frontier:** In this small, corridor-like maze the maximum frontier stays between 2 and 4 for both algorithms and does not grow with distance. BFS's frontier was greater than or equal to DFS's in every pair tested (for example 4 vs 3 for (3,2)→(0,0)). Frontier growth would be much more visible in a large open grid, where BFS stores an entire ring of states at the current depth while DFS stores only the pending siblings along one path.
8. **Same path for both?** Yes, for (0,0)→(0,2): both return the straight path along the top row. This happens when only one short route exists between the two points, so there is no choice for the two strategies to disagree on. Where the maze offers alternatives (a loop around the wall block), BFS and DFS diverge.
9. **Unreachable goal:** Both return `(None, max_frontier)`. Each algorithm exhausts every reachable state, the frontier empties, and the `while` loop ends without finding the goal. This is the correct "no solution" behavior, and the `visited` set guarantees it terminates.

## 10. Where This Is Used
**BFS:** shortest paths in unweighted graphs, network exploration, web crawling. **DFS:** cycle detection, topological sorting, backtracking, file-system traversal. Both are *uninformed*: they have no estimate of which direction is closer to the goal. Informed search (A*) adds a heuristic for that.

## 11. Conclusion

The only code difference between the two algorithms is `frontier.popleft()` (BFS, FIFO) versus `frontier.pop()` (DFS, LIFO), and that single line decides their behavior.

| | BFS | DFS |
|---|---|---|
| Shortest path (equal move costs) | Always | Not guaranteed (10 vs 4 moves for (0,0)→(2,2)) |
| Path depends on neighbor order | Route only, never length | Both route and length |
| Max frontier in these tests | ≥ DFS in every test | ≤ BFS in every test |
| Goal unreachable | Returns `None` | Returns `None` |

On the main maze (0,0)→(4,4) the two tie at 8 moves because two equal-length routes exist. On the other tested pairs, DFS found a longer path in three of four cases (10 vs 4, 9 vs 5, 10 vs 8). So BFS is the right choice when the shortest path matters, while DFS's smaller frontier is its main advantage in memory use.